# 🔬 Phase 4 — Controlled Class Imbalance: Sweep & Benchmark Data Generation

**Project**: Low-Resource Language (Ukrainian) Sentiment & Emotion Analysis  
**Dataset**: [KSE-RESEARCH-Group/UAReviews](https://huggingface.co/datasets/KSE-RESEARCH-Group/UAReviews) (11,580 Ukrainian reviews)  
**Embeddings**: Pre-encoded 1024-d dense vectors from [Qwen/Qwen3-Embedding-0.6B](https://huggingface.co/Qwen/Qwen3-Embedding-0.6B)  
**Hardware**: Torch-CUDA GPU Runtime  

### Experimental Focus:
This notebook handles the **experimental sweep execution** and **benchmark dataset generation**:
1. **Oversampling Operators**: Pure NumPy implementations of `duplication`, classic `smote`, norm-preserving `smote_renorm`, and retrieval-based `embsmote` (Inoshita, 2026).
2. **Controlled Sweep**: Imbalance ratios $\rho \in \{0.10, 0.25, 0.50, 0.75, 1.00\}$ across `LinearSVC` and `TorchMLPClassifier`.
3. **Dual Evaluation**: In-distribution `test` set (1,737) and out-of-distribution `challenge` set (1,737).
4. **Resumable Disk Checkpointing**: Every condition is saved as an independent JSON file in `cache/phase4/`.
5. **Benchmark Summary**: Results are aggregated into `cache/phase4/phase4_sweep_summary.csv`.

> **Note**: All plotting, gain curves, robustness trade-off analysis, and publication figures have been separated into **`phase_4_analysis.ipynb`**.


In [ ]:
# 1. Setup & GPU Verification
!pip install -q scikit-learn matplotlib seaborn torch pandas numpy tqdm

import os
import gc
import json
import time
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, f1_score, accuracy_score, precision_score, recall_score, confusion_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.svm import LinearSVC
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"PyTorch Runtime: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


In [ ]:
# 2. Storage Setup & Load Phase 1-3 Artifacts
try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/lrl_sentiment_short'
except Exception:
    BASE_DIR = os.path.abspath('./lrl_cache')

CACHE_DIR = os.path.join(BASE_DIR, 'cache')
EMBED_DIR = os.path.join(CACHE_DIR, 'embeddings')
PHASE4_DIR = os.path.join(CACHE_DIR, 'phase4')
os.makedirs(PHASE4_DIR, exist_ok=True)

with open(os.path.join(CACHE_DIR, 'label_encoder.json'), 'r', encoding='utf-8') as f:
    label_map = json.load(f)
emotion2id = label_map['emotion2id']
id2emotion = {int(k): v for k, v in label_map['id2emotion'].items()}
EMOTION_LABELS = [id2emotion[i] for i in range(len(id2emotion))]
NUM_CLASSES = len(EMOTION_LABELS)

with open(os.path.join(CACHE_DIR, 'split_indices.json'), 'r') as f:
    split_indices = json.load(f)

train_idx = np.array(split_indices['train'])
test_idx = np.array(split_indices['test'])
chal_idx = np.array(split_indices['challenge'])

df = pd.read_parquet(os.path.join(CACHE_DIR, 'ua_reviews_clean.parquet'))
y_all = df['label'].values
y_train = y_all[train_idx]
y_test = y_all[test_idx]
y_chal = y_all[chal_idx]

all_emb_path = os.path.join(EMBED_DIR, 'qwen3_embeddings_all.npy')
if os.path.exists(all_emb_path):
    X_all = np.load(all_emb_path)
    X_train = X_all[train_idx]
    X_test = X_all[test_idx]
    X_chal = X_all[chal_idx]
else:
    X_train = np.load(os.path.join(EMBED_DIR, 'qwen3_embeddings_train.npy'))
    X_test = np.load(os.path.join(EMBED_DIR, 'qwen3_embeddings_test.npy'))
    X_chal = np.load(os.path.join(EMBED_DIR, 'qwen3_embeddings_challenge.npy'))

print(f"Loaded artifacts:")
print(f"  X_train: {X_train.shape} | y_train: {y_train.shape}")
print(f"  X_test:  {X_test.shape}  | y_test:  {y_test.shape}")
print(f"  X_chal:  {X_chal.shape}  | y_chal:  {y_chal.shape}")
print(f"  Classes: {EMOTION_LABELS}")


## 3. Oversampling Strategies Implementation

We implement 4 distinct data augmentation mechanisms from scratch in pure NumPy/Scikit-learn, operating on normalized 1024-d embeddings:
1. **`duplication`** (Random Oversampling): Sample minority instances uniformly at random with replacement.
2. **`smote`** (Classic Embedding SMOTE): Intra-class linear interpolation along the segment joining a minority instance to a random $k$-nearest neighbor:
   $$z_{\text{synth}} = z_i + \lambda(z_{nn} - z_i), \quad \lambda \sim U(0, 1)$$
3. **`smote_renorm`** (L2-Renormalized SMOTE): Synthetic vectors from SMOTE are re-projected onto the unit hypersphere:
   $$z_{\text{renorm}} = \frac{z_{\text{synth}}}{\|z_{\text{synth}}\|_2}$$
   This counteracts norm-shrinkage inherent in linear interpolation of high-dimensional directional vectors.
4. **`embsmote`** (Retrieval-Based EmbSMOTE; Inoshita 2026): Intra-class interpolation is computed, followed by nearest-neighbor retrieval from the real training set points of that class:
   $$x^* = \arg\min_{x \in \mathcal{D}_{\text{train}}^{(c)}} \|z_{\text{synth}} - x\|$$
   This structurally guarantees that every synthetic sample is an authentic member of the class manifold.


In [ ]:
# 3. Oversampling Operators
SMOTE_K = 5

def oversample_duplication(X_c, n_synth, rng):
    # Random oversampling with replacement
    if n_synth <= 0 or len(X_c) == 0:
        return np.empty((0, X_c.shape[1]), dtype=np.float32)
    indices = rng.choice(len(X_c), size=n_synth, replace=True)
    return X_c[indices].copy()

def oversample_smote(X_c, n_synth, k=SMOTE_K, rng=None, renorm=False):
    # Classic SMOTE interpolation in embedding space
    if n_synth <= 0 or len(X_c) == 0:
        return np.empty((0, X_c.shape[1]), dtype=np.float32)
    
    n_samples, d = X_c.shape
    k_eff = max(1, min(k, n_samples - 1))
    
    if n_samples <= 1:
        return oversample_duplication(X_c, n_synth, rng)
    
    nbrs = NearestNeighbors(n_neighbors=k_eff + 1, metric='cosine').fit(X_c)
    _, indices = nbrs.kneighbors(X_c)
    
    base_idx = rng.choice(n_samples, size=n_synth, replace=True)
    neighbor_col = rng.integers(1, k_eff + 1, size=n_synth)
    chosen_nn_idx = indices[base_idx, neighbor_col]
    
    lambdas = rng.uniform(0.0, 1.0, size=(n_synth, 1)).astype(np.float32)
    diff = X_c[chosen_nn_idx] - X_c[base_idx]
    synth = X_c[base_idx] + lambdas * diff
    
    if renorm:
        norms = np.linalg.norm(synth, axis=1, keepdims=True)
        norms = np.maximum(norms, 1e-8)
        synth = synth / norms
        
    return synth.astype(np.float32)

def oversample_embsmote(X_c, n_synth, k=SMOTE_K, rng=None):
    # EmbSMOTE (Inoshita, 2026): Intra-class interpolation + nearest-neighbor retrieval
    if n_synth <= 0 or len(X_c) == 0:
        return np.empty((0, X_c.shape[1]), dtype=np.float32)
    
    # 1. Generate interpolated candidate vectors
    candidates = oversample_smote(X_c, n_synth, k=k, rng=rng, renorm=True)
    
    # 2. Retrieve nearest real instance in X_c for each candidate
    nbrs = NearestNeighbors(n_neighbors=1, metric='cosine').fit(X_c)
    _, ret_indices = nbrs.kneighbors(candidates)
    
    retrieved = X_c[ret_indices.squeeze(1)].copy()
    return retrieved.astype(np.float32)

def build_oversampled_dataset(X_tr, y_tr, strategy, rho, rng=None):
    # Builds augmented training dataset where minority classes are oversampled 
    # to reach target size: target_size(c) = max(N_c, round(rho * N_majority)).
    if rng is None:
        rng = np.random.default_rng(SEED)
        
    class_counts = {c: int((y_tr == c).sum()) for c in range(NUM_CLASSES)}
    maj_count = max(class_counts.values())
    
    X_parts = []
    y_parts = []
    added_stats = {}
    
    for c in range(NUM_CLASSES):
        X_c = X_tr[y_tr == c]
        n_orig = len(X_c)
        X_parts.append(X_c)
        y_parts.append(np.full(n_orig, c, dtype=np.int64))
        
        if strategy == "none":
            added_stats[c] = 0
            continue
            
        target_count = max(n_orig, int(round(rho * maj_count)))
        n_synth = target_count - n_orig
        
        if n_synth > 0:
            if strategy == "duplication":
                X_synth = oversample_duplication(X_c, n_synth, rng)
            elif strategy == "smote":
                X_synth = oversample_smote(X_c, n_synth, k=SMOTE_K, rng=rng, renorm=False)
            elif strategy == "smote_renorm":
                X_synth = oversample_smote(X_c, n_synth, k=SMOTE_K, rng=rng, renorm=True)
            elif strategy == "embsmote":
                X_synth = oversample_embsmote(X_c, n_synth, k=SMOTE_K, rng=rng)
            else:
                raise ValueError(f"Unknown strategy: {strategy}")
                
            X_parts.append(X_synth)
            y_parts.append(np.full(n_synth, c, dtype=np.int64))
            added_stats[c] = int(n_synth)
        else:
            added_stats[c] = 0
            
    X_aug = np.vstack(X_parts).astype(np.float32)
    y_aug = np.concatenate(y_parts).astype(np.int64)
    
    perm = rng.permutation(len(y_aug))
    return X_aug[perm], y_aug[perm], added_stats


## 4. Classifiers Adapted for Torch-CUDA

We evaluate two models:
1. **`LinearSVC`**: Linear Support Vector Classifier with One-vs-Rest multi-class strategy ($C=1.0$).
2. **`TorchMLPClassifier`**: Fully GPU-accelerated multi-layer perceptron (Linear -> BatchNorm -> ReLU -> Dropout -> Linear -> BatchNorm -> ReLU -> Dropout -> Linear), trained with Cross-Entropy Loss, AdamW optimizer, and early stopping on a stratified validation set.


In [ ]:
# 4. Classifier Definitions
import torch.nn as nn

class TorchMLP(nn.Module):
    def __init__(self, in_dim=1024, hidden_dims=(256, 64), num_classes=NUM_CLASSES, dropout=0.2):
        super().__init__()
        layers = []
        d = in_dim
        for h in hidden_dims:
            layers.append(nn.Linear(d, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            d = h
        layers.append(nn.Linear(d, num_classes))
        self.net = nn.Sequential(*layers)
        
    def forward(self, x):
        return self.net(x)

class TorchMLPClassifier:
    # sklearn-compatible wrapper for GPU-accelerated PyTorch MLP with early stopping & class-weight support
    def __init__(self, hidden_dims=(256, 64), lr=1e-3, weight_decay=1e-4, batch_size=128,
                 epochs=60, patience=8, random_state=SEED, class_weight=None):
        self.hidden_dims = hidden_dims
        self.lr = lr
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.epochs = epochs
        self.patience = patience
        self.random_state = random_state
        self.class_weight = class_weight
        self.model = None

    def fit(self, X, y, X_val=None, y_val=None):
        torch.manual_seed(self.random_state)
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.int64)
        
        # If validation fold is not provided, carve 10% stratified split
        if X_val is None or y_val is None:
            from sklearn.model_selection import train_test_split
            X_tr, X_va, y_tr, y_va = train_test_split(
                X, y, test_size=0.10, random_state=self.random_state, stratify=y
            )
        else:
            X_tr, y_tr, X_va, y_va = X, y, X_val, y_val

        ds_tr = TensorDataset(torch.as_tensor(X_tr, dtype=torch.float32), torch.as_tensor(y_tr, dtype=torch.long))
        loader = DataLoader(ds_tr, batch_size=self.batch_size, shuffle=True)
        
        X_va_t = torch.as_tensor(X_va, dtype=torch.float32, device=DEVICE)
        y_va_t = torch.as_tensor(y_va, dtype=torch.long, device=DEVICE)

        self.model = TorchMLP(in_dim=X.shape[1], hidden_dims=self.hidden_dims, num_classes=NUM_CLASSES).to(DEVICE)
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=self.weight_decay)
        
        if self.class_weight == 'balanced':
            counts = np.bincount(y_tr, minlength=NUM_CLASSES).astype(float)
            w = len(y_tr) / (NUM_CLASSES * np.maximum(counts, 1.0))
            loss_fn = nn.CrossEntropyLoss(weight=torch.as_tensor(w, dtype=torch.float32, device=DEVICE))
        else:
            loss_fn = nn.CrossEntropyLoss()

        best_score, best_state, wait = -1.0, None, 0
        for epoch in range(1, self.epochs + 1):
            self.model.train()
            for bx, by in loader:
                bx, by = bx.to(DEVICE), by.to(DEVICE)
                opt.zero_grad(set_to_none=True)
                loss = loss_fn(self.model(bx), by)
                loss.backward()
                opt.step()

            self.model.eval()
            with torch.no_grad():
                va_preds = self.model(X_va_t).argmax(dim=-1).cpu().numpy()
                va_f1 = f1_score(y_va, va_preds, average='macro', zero_division=0)

            if va_f1 > best_score + 1e-4:
                best_score = va_f1
                wait = 0
                best_state = {k: v.cpu().clone() for k, v in self.model.state_dict().items()}
            else:
                wait += 1
                if wait >= self.patience:
                    break

        if best_state is not None:
            self.model.load_state_dict(best_state)
            self.model.to(DEVICE)
        return self

    @torch.no_grad()
    def predict(self, X):
        self.model.eval()
        X_t = torch.as_tensor(X, dtype=torch.float32)
        ds = TensorDataset(X_t)
        loader = DataLoader(ds, batch_size=256, shuffle=False)
        preds = []
        for (bx,) in loader:
            p = self.model(bx.to(DEVICE)).argmax(dim=-1).cpu().numpy()
            preds.append(p)
        return np.concatenate(preds)


## 5. Sweep Driver with Resumable Disk Checkpointing

We sweep across:
- **Strategies**: `['none', 'duplication', 'smote', 'smote_renorm', 'embsmote']`
- **Target Imbalance Ratios ($\rho$)**: `[0.10, 0.25, 0.50, 0.75, 1.00]`
- **Classifiers**: `['linearsvc', 'mlp']`

Every condition result is cached in `cache/phase4/{cond_id}.json`. If execution is paused or interrupted, reruns skip existing conditions instantly.


In [ ]:
# 5. Sweep Driver Implementation
STRATEGIES = ["none", "duplication", "smote", "smote_renorm", "embsmote"]
RHOS = [0.10, 0.25, 0.50, 0.75, 1.00]
CLASSIFIERS = ["linearsvc", "mlp"]

def score_predictions(y_true, y_pred):
    return {
        "macro_f1": float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
        "weighted_f1": float(f1_score(y_true, y_pred, average="weighted", zero_division=0)),
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "per_class_f1": {EMOTION_LABELS[i]: float(f) for i, f in enumerate(f1_score(y_true, y_pred, average=None, zero_division=0))},
        "confusion_matrix": confusion_matrix(y_true, y_pred, labels=list(range(NUM_CLASSES))).tolist()
    }

def run_condition(strategy, rho, clf_name, seed=SEED):
    cond_id = f"strat_{strategy}_rho{rho}__{clf_name}"
    out_path = os.path.join(PHASE4_DIR, f"{cond_id}.json")
    
    if os.path.exists(out_path):
        with open(out_path, 'r') as f:
            cached = json.load(f)
        if 'test_preds' in cached and 'chal_preds' in cached:
            print(f"[cached] {cond_id} -> Test Macro F1: {cached['test_metrics']['macro_f1']:.4f}")
            return cached

    t0 = time.time()
    rng = np.random.default_rng(seed)
    
    if clf_name == "linearsvc":
        # LinearSVC trains on full training set with oversampling
        if strategy == "none":
            X_train_c, y_train_c = X_train.copy(), y_train.copy()
            added_stats = {c: 0 for c in range(NUM_CLASSES)}
        else:
            X_train_c, y_train_c, added_stats = build_oversampled_dataset(
                X_train, y_train, strategy=strategy, rho=rho, rng=rng
            )
        model = LinearSVC(C=1.0, max_iter=5000, random_state=seed)
        model.fit(X_train_c, y_train_c)
        final_train_size = len(X_train_c)
    elif clf_name == "mlp":
        # CRITICAL INTEGRITY: Carve 10% stratified validation split from raw unaugmented X_train PRIOR to oversampling
        # to prevent synthetic and duplicate data from leaking into early stopping validation
        from sklearn.model_selection import train_test_split
        X_tr_raw, X_va, y_tr_raw, y_va = train_test_split(
            X_train, y_train, test_size=0.10, random_state=seed, stratify=y_train
        )
        if strategy == "none":
            X_tr_aug, y_tr_aug = X_tr_raw.copy(), y_tr_raw.copy()
            added_stats = {c: 0 for c in range(NUM_CLASSES)}
        else:
            X_tr_aug, y_tr_aug, added_stats = build_oversampled_dataset(
                X_tr_raw, y_tr_raw, strategy=strategy, rho=rho, rng=rng
            )
        model = TorchMLPClassifier(random_state=seed)
        model.fit(X_tr_aug, y_tr_aug, X_val=X_va, y_val=y_va)
        final_train_size = len(X_tr_aug)
    else:
        raise ValueError(clf_name)
        
    n_synth_total = sum(added_stats.values())
    print(f"[{cond_id}] Train size: {final_train_size} (Added synthetics: {n_synth_total})")

    test_preds = model.predict(X_test)
    chal_preds = model.predict(X_chal)
    
    test_metrics = score_predictions(y_test, test_preds)
    chal_metrics = score_predictions(y_chal, chal_preds)
    elapsed = time.time() - t0

    result = {
        "strategy": strategy,
        "rho": rho,
        "clf": clf_name,
        "train_size": int(final_train_size),
        "n_synthetics": int(n_synth_total),
        "synthetics_per_class": added_stats,
        "elapsed_sec": float(elapsed),
        "test_metrics": test_metrics,
        "challenge_metrics": chal_metrics,
        "test_preds": test_preds.tolist() if hasattr(test_preds, 'tolist') else list(test_preds),
        "chal_preds": chal_preds.tolist() if hasattr(chal_preds, 'tolist') else list(chal_preds)
    }
    
    with open(out_path, 'w') as f:
        json.dump(result, f, indent=2)
        
    print(f"[done] {cond_id} | Test Macro F1: {test_metrics['macro_f1']:.4f} | "
          f"Chal Macro F1: {chal_metrics['macro_f1']:.4f} | Time: {elapsed:.1f}s")
    
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
        
    return result


In [ ]:
# Cost-Sensitive Baselines: Inverse-Frequency Class Weighting on Raw Training Data
print("="*80)
print("⚖️ Evaluating Cost-Sensitive Baselines (class_weight='balanced')")
print("="*80)

# 1. LinearSVC with class_weight='balanced'
svc_balanced = LinearSVC(C=1.0, class_weight='balanced', max_iter=5000, random_state=SEED)
svc_balanced.fit(X_train, y_train)
p_test_svcb = svc_balanced.predict(X_test)
p_chal_svcb = svc_balanced.predict(X_chal)
m_test_svcb = score_predictions(y_test, p_test_svcb)
m_chal_svcb = score_predictions(y_chal, p_chal_svcb)
print(f"LinearSVC (balanced) | Test Macro F1: {m_test_svcb['macro_f1']:.4f} (Acc: {m_test_svcb['accuracy']*100:.2f}%) | "
      f"Chal Macro F1: {m_chal_svcb['macro_f1']:.4f} (Acc: {m_chal_svcb['accuracy']*100:.2f}%)")

# 2. TorchMLP with Weighted Cross-Entropy Loss (carving val fold prior to training)
from sklearn.model_selection import train_test_split
X_tr_b, X_va_b, y_tr_b, y_va_b = train_test_split(
    X_train, y_train, test_size=0.10, random_state=SEED, stratify=y_train
)
mlp_balanced = TorchMLPClassifier(random_state=SEED, class_weight='balanced')
mlp_balanced.fit(X_tr_b, y_tr_b, X_val=X_va_b, y_val=y_va_b)
p_test_mlpb = mlp_balanced.predict(X_test)
p_chal_mlpb = mlp_balanced.predict(X_chal)
m_test_mlpb = score_predictions(y_test, p_test_mlpb)
m_chal_mlpb = score_predictions(y_chal, p_chal_mlpb)
print(f"TorchMLP  (balanced) | Test Macro F1: {m_test_mlpb['macro_f1']:.4f} (Acc: {m_test_mlpb['accuracy']*100:.2f}%) | "
      f"Chal Macro F1: {m_chal_mlpb['macro_f1']:.4f} (Acc: {m_chal_mlpb['accuracy']*100:.2f}%)")


In [ ]:
# 6. Execute Experimental Sweep
all_results = []

# Baseline conditions (none) at rho=1.0
for clf in CLASSIFIERS:
    all_results.append(run_condition("none", 1.00, clf))

# Sweep oversampling strategies across rhos
for rho in RHOS:
    for strategy in ["duplication", "smote", "smote_renorm", "embsmote"]:
        for clf in CLASSIFIERS:
            all_results.append(run_condition(strategy, rho, clf))

print(f"\nSweep Complete! {len(all_results)} total conditions evaluated.")


## 6. Benchmark Data Generation & Summary CSV Compilation

Consolidates all cached JSON condition checkpoints into a single structured benchmark dataset `phase4_sweep_summary.csv`.


In [ ]:
# 7. Aggregate Results into DataFrame
if 'all_results' not in globals() or not all_results:
    all_results = []
    # Baseline conditions (none) at rho=1.0
    for clf in CLASSIFIERS:
        all_results.append(run_condition("none", 1.00, clf))

    # Sweep oversampling strategies across rhos
    for rho in RHOS:
        for strategy in ["duplication", "smote", "smote_renorm", "embsmote"]:
            for clf in CLASSIFIERS:
                all_results.append(run_condition(strategy, rho, clf))

rows = []
for r in all_results:
    strat = r['strategy']
    rho = r['rho']
    clf = r['clf']
    tm = r['test_metrics']
    cm = r['challenge_metrics']
    
    row = {
        'Strategy': strat,
        'Rho': rho,
        'Classifier': 'LinearSVC' if clf == 'linearsvc' else 'TorchMLP',
        'Train Size': r['train_size'],
        'Test Macro F1': tm['macro_f1'],
        'Test Weighted F1': tm['weighted_f1'],
        'Test Acc': tm['accuracy'],
        'Chal Macro F1': cm['macro_f1'],
        'Chal Weighted F1': cm['weighted_f1'],
        'Fear F1': tm['per_class_f1']['Fear'],
        'Surprise F1': tm['per_class_f1']['Surprise'],
        'Disgust F1': tm['per_class_f1']['Disgust'],
        'Sadness F1': tm['per_class_f1']['Sadness'],
        'Chal Fear F1': cm['per_class_f1']['Fear'],
        'Chal Surprise F1': cm['per_class_f1']['Surprise'],
        'Chal Disgust F1': cm['per_class_f1']['Disgust'],
        'Chal Sadness F1': cm['per_class_f1']['Sadness'],
        'Elapsed (s)': r['elapsed_sec']
    }
    rows.append(row)

results_df = pd.DataFrame(rows)
summary_csv = os.path.join(PHASE4_DIR, 'phase4_sweep_summary.csv')
results_df.to_csv(summary_csv, index=False)
print(f"Summary saved: {summary_csv}")
print(results_df.head(10))


## 7. Export Inference Predictions for Statistical Significance (Figures 8 & 9)

Assembles and serializes sample-level inference prediction vectors across In-Distribution Test ($N=1{,}737$) and Out-of-Distribution Challenge ($N=1{,}737$) sets for:
- **`LinearSVC`**: Baseline ($ho=1.0$) and all 4 oversampling strategies at champion $ho=0.25$.
- **`TorchMLP`**: Baseline ($ho=1.0$) and all 4 oversampling strategies at champion $ho=0.10$.

These artifacts (`predictions_linearsvc_rho025.json` and `predictions_mlp_rho01.json`) are directly consumed by **McNemar's Test** (Figure 8) and **Stratified Bootstrap Confidence Intervals** (Figure 9) in `phase_4_analysis.ipynb`.


In [ ]:
# 8. Export Champion Predictions for McNemar & Stratified Bootstrap
pred_lookup = {}
for r in all_results:
    key = (r['clf'], r['strategy'], float(r['rho']))
    if 'test_preds' in r and 'chal_preds' in r:
        pred_lookup[key] = {
            'test_preds': r['test_preds'],
            'chal_preds': r['chal_preds'],
            'test_macro_f1': r['test_metrics']['macro_f1'],
            'chal_macro_f1': r['challenge_metrics']['macro_f1']
        }

# Assemble LinearSVC package (champion rho=0.25, plus baseline none at rho=1.0)
svc_strats = {}
for s in ["none", "duplication", "smote", "smote_renorm", "embsmote"]:
    r_key = 1.0 if s == "none" else 0.25
    if ('linearsvc', s, r_key) in pred_lookup:
        svc_strats[s] = pred_lookup[('linearsvc', s, r_key)]
    else:
        cond_res = run_condition(s, r_key, 'linearsvc')
        svc_strats[s] = {
            'test_preds': cond_res['test_preds'],
            'chal_preds': cond_res['chal_preds'],
            'test_macro_f1': cond_res['test_metrics']['macro_f1'],
            'chal_macro_f1': cond_res['challenge_metrics']['macro_f1']
        }

svc_payload = {
    'y_test': y_test.tolist() if hasattr(y_test, 'tolist') else list(y_test),
    'y_chal': y_chal.tolist() if hasattr(y_chal, 'tolist') else list(y_chal),
    'linearsvc': svc_strats
}

# Assemble TorchMLP package (champion rho=0.10, plus baseline none at rho=1.0)
mlp_strats = {}
for s in ["none", "duplication", "smote", "smote_renorm", "embsmote"]:
    r_key = 1.0 if s == "none" else 0.10
    if ('mlp', s, r_key) in pred_lookup:
        mlp_strats[s] = pred_lookup[('mlp', s, r_key)]
    else:
        cond_res = run_condition(s, r_key, 'mlp')
        mlp_strats[s] = {
            'test_preds': cond_res['test_preds'],
            'chal_preds': cond_res['chal_preds'],
            'test_macro_f1': cond_res['test_metrics']['macro_f1'],
            'chal_macro_f1': cond_res['challenge_metrics']['macro_f1']
        }

mlp_payload = {
    'y_test': y_test.tolist() if hasattr(y_test, 'tolist') else list(y_test),
    'y_chal': y_chal.tolist() if hasattr(y_chal, 'tolist') else list(y_chal),
    'mlp': mlp_strats
}

# Save to PHASE4_DIR and local ./phase4/
out_svc_path = os.path.join(PHASE4_DIR, 'predictions_linearsvc_rho025.json')
out_mlp_path = os.path.join(PHASE4_DIR, 'predictions_mlp_rho01.json')

with open(out_svc_path, 'w') as f:
    json.dump(svc_payload, f)
with open(out_mlp_path, 'w') as f:
    json.dump(mlp_payload, f)

if os.path.abspath('phase4') != os.path.abspath(PHASE4_DIR):
    os.makedirs('phase4', exist_ok=True)
    with open('phase4/predictions_linearsvc_rho025.json', 'w') as f1:
        json.dump(svc_payload, f1)
    with open('phase4/predictions_mlp_rho01.json', 'w') as f2:
        json.dump(mlp_payload, f2)

print(f"Exported champion inference prediction artifacts:")
print(f"  {out_svc_path} (LinearSVC rho=0.25)")
print(f"  {out_mlp_path} (TorchMLP rho=0.10)")


## 8. Multi-Seed Stochasticity Evaluation: TorchMLP Robustness & Variance (Mean ± SD)

To address the methodological caveat that single-seed checkpointing leaves neural training-procedure stochasticity unmodeled (random weight initialization, minibatch shuffling, dropout masks, validation split sampling, and SMOTE random neighbor selection), this section executes a controlled multi-seed sweep across 5 diverse random seeds ($S \in \{42, 101, 202, 303, 404\}$).

### Objectives:
1. **Quantify Stochasticity**: Evaluate `TorchMLPClassifier` across multiple random seeds for Baseline ($\rho=1.00$) versus oversampling strategies ($\rho=0.10$).
2. **Report Mean ± SD**: Calculate empirical mean and sample standard deviation ($\text{Mean} \pm \text{SD}$) for Macro F1, Weighted F1, Accuracy, and per-class emotion F1 scores.
3. **Paired Hypothesis Testing**: Compute paired seed differences ($\Delta = \text{Metric}_{\text{strategy}}^{(s)} - \text{Metric}_{\text{none}}^{(s)}$) isolating resampling effects from seed variance.
4. **Holm-Bonferroni Multiplicity Control**: Apply step-down Holm-Bonferroni correction ($p_{\text{Holm}}$) across evaluated strategies ($m=4$) and per-class emotion categories ($m=7$ all classes, $m=4$ tail classes).

In [ ]:
# 9. Multi-Seed Stochasticity Sweep & Holm-Bonferroni Significance for TorchMLP
import os
import gc
import json
import time
import random
import numpy as np
import pandas as pd
import scipy.stats as stats
import torch
from sklearn.metrics import f1_score, accuracy_score, precision_score, recall_score, confusion_matrix

MLP_SEEDS = [42, 101, 202, 303, 404]
MULTISEED_CONDITIONS = [
    ("none", 1.00),
    ("duplication", 0.10),
    ("smote", 0.10),
    ("smote_renorm", 0.10),
    ("embsmote", 0.10)
]

FORCE_RERUN_SEEDS = False  # Set to True on Colab to force fresh GPU re-training across all seeds

def holm_bonferroni(pvals):
    pvals = np.asarray(pvals, dtype=float)
    order = np.argsort(pvals)
    adj = np.empty_like(pvals)
    running = 0.0
    for rank, i in enumerate(order):
        running = max(running, (len(pvals) - rank) * pvals[i])
        adj[i] = min(1.0, running)
    return adj

def sig_stars(p):
    if p < 0.001: return '***'
    if p < 0.01: return '**'
    if p < 0.05: return '*'
    return 'ns'

def run_mlp_seed(strategy, rho, seed, force_rerun=FORCE_RERUN_SEEDS):
    cond_id = f"strat_{strategy}_rho{rho}__mlp_seed{seed}"
    out_path = os.path.join(PHASE4_DIR, f"{cond_id}.json")
    local_path = os.path.join('phase4', f"{cond_id}.json")
    
    if not force_rerun:
        for p in [out_path, local_path]:
            if os.path.exists(p):
                with open(p, 'r') as f:
                    cached = json.load(f)
                if 'test_metrics' in cached and 'challenge_metrics' in cached:
                    return cached
                    
        # If seed 42, check if single-seed sweep checkpoint already exists
        if seed == SEED:
            for p in [os.path.join(PHASE4_DIR, f"strat_{strategy}_rho{rho}__mlp.json"),
                      os.path.join('phase4', f"strat_{strategy}_rho{rho}__mlp.json")]:
                if os.path.exists(p):
                    with open(p, 'r') as f:
                        cached = json.load(f)
                    if 'test_metrics' in cached and 'challenge_metrics' in cached:
                        cached['seed'] = seed
                        with open(out_path, 'w') as f_out:
                            json.dump(cached, f_out, indent=2)
                        return cached

    # Train model on device
    t0 = time.time()
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    rng = np.random.default_rng(seed)
    # Validation split carved prior to oversampling to prevent duplicate/synthetic leakage
    from sklearn.model_selection import train_test_split
    X_tr_raw, X_va, y_tr_raw, y_va = train_test_split(
        X_train, y_train, test_size=0.10, random_state=seed, stratify=y_train
    )
    if strategy == "none":
        X_tr_aug, y_tr_aug = X_tr_raw.copy(), y_tr_raw.copy()
        added_stats = {c: 0 for c in range(NUM_CLASSES)}
    else:
        X_tr_aug, y_tr_aug, added_stats = build_oversampled_dataset(
            X_tr_raw, y_tr_raw, strategy=strategy, rho=rho, rng=rng
        )

    n_synth_total = sum(added_stats.values())
    model = TorchMLPClassifier(random_state=seed)
    model.fit(X_tr_aug, y_tr_aug, X_val=X_va, y_val=y_va)
    X_train_c = X_tr_aug

    test_preds = model.predict(X_test)
    chal_preds = model.predict(X_chal)
    test_metrics = score_predictions(y_test, test_preds)
    chal_metrics = score_predictions(y_chal, chal_preds)
    elapsed = time.time() - t0

    result = {
        "strategy": strategy,
        "rho": rho,
        "seed": seed,
        "clf": "mlp",
        "train_size": int(len(X_train_c)),
        "n_synthetics": int(n_synth_total),
        "synthetics_per_class": added_stats,
        "elapsed_sec": float(elapsed),
        "test_metrics": test_metrics,
        "challenge_metrics": chal_metrics,
        "test_preds": test_preds.tolist() if hasattr(test_preds, 'tolist') else list(test_preds),
        "chal_preds": chal_preds.tolist() if hasattr(chal_preds, 'tolist') else list(chal_preds)
    }

    with open(out_path, 'w') as f:
        json.dump(result, f, indent=2)
    if os.path.abspath('phase4') != os.path.abspath(PHASE4_DIR):
        os.makedirs('phase4', exist_ok=True)
        with open(local_path, 'w') as f_loc:
            json.dump(result, f_loc, indent=2)

    print(f"[{cond_id}] Test F1: {test_metrics['macro_f1']:.4f} | Chal F1: {chal_metrics['macro_f1']:.4f} ({elapsed:.1f}s)")
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return result

# Execute Multi-Seed Runs
print(f"--- Running Multi-Seed Stochasticity Evaluation (Seeds: {MLP_SEEDS}) ---")
multiseed_records = []
for strat, rho in MULTISEED_CONDITIONS:
    for s in MLP_SEEDS:
        res = run_mlp_seed(strat, rho, s)
        tm = res['test_metrics']
        cm = res['challenge_metrics']
        rec = {
            'strategy': strat,
            'rho': rho,
            'seed': s,
            'test_macro_f1': tm['macro_f1'],
            'test_weighted_f1': tm['weighted_f1'],
            'test_acc': tm['accuracy'],
            'chal_macro_f1': cm['macro_f1'],
            'chal_weighted_f1': cm['weighted_f1'],
            'chal_acc': cm['accuracy']
        }
        for emo in EMOTION_LABELS:
            rec[f'test_f1_{emo}'] = tm['per_class_f1'][emo]
            rec[f'chal_f1_{emo}'] = cm['per_class_f1'][emo]
        multiseed_records.append(rec)

multiseed_df = pd.DataFrame(multiseed_records)
all_runs_csv = os.path.join(PHASE4_DIR, 'torchmlp_multiseed_all_runs.csv')
multiseed_df.to_csv(all_runs_csv, index=False)
if os.path.abspath('phase4') != os.path.abspath(PHASE4_DIR):
    multiseed_df.to_csv('phase4/torchmlp_multiseed_all_runs.csv', index=False)

# Compute Strategy Mean ± SD and Paired Differences vs Baseline
base_df = multiseed_df[multiseed_df['strategy'] == 'none'].set_index('seed')
strategies_to_test = [s for s, _ in MULTISEED_CONDITIONS if s != 'none']
raw_p_chal, raw_p_test = [], []
strat_deltas_chal, strat_deltas_test = {}, {}

for s in strategies_to_test:
    sub = multiseed_df[multiseed_df['strategy'] == s].set_index('seed')
    d_c = sub['chal_macro_f1'] - base_df['chal_macro_f1']
    d_t = sub['test_macro_f1'] - base_df['test_macro_f1']
    strat_deltas_chal[s] = d_c.values
    strat_deltas_test[s] = d_t.values
    
    t_c, p_c = stats.ttest_rel(sub['chal_macro_f1'], base_df['chal_macro_f1'])
    t_t, p_t = stats.ttest_rel(sub['test_macro_f1'], base_df['test_macro_f1'])
    raw_p_chal.append(p_c)
    raw_p_test.append(p_t)

holm_chal = holm_bonferroni(raw_p_chal)
holm_test = holm_bonferroni(raw_p_test)

strat_lookup = {}
for idx, s in enumerate(strategies_to_test):
    strat_lookup[s] = {
        'chal_raw_p': float(raw_p_chal[idx]),
        'chal_holm_p': float(holm_chal[idx]),
        'test_raw_p': float(raw_p_test[idx]),
        'test_holm_p': float(holm_test[idx]),
        'chal_delta_mean': float(np.mean(strat_deltas_chal[s])),
        'chal_delta_sd': float(np.std(strat_deltas_chal[s], ddof=1)),
        'test_delta_mean': float(np.mean(strat_deltas_test[s])),
        'test_delta_sd': float(np.std(strat_deltas_test[s], ddof=1))
    }

summary_rows = []
for (s, rho) in MULTISEED_CONDITIONS:
    sub = multiseed_df[multiseed_df['strategy'] == s]
    row = {
        'Strategy': s,
        'Rho': rho,
        'Chal_Macro_F1': f"{sub['chal_macro_f1'].mean():.4f} ± {sub['chal_macro_f1'].std(ddof=1):.4f}",
        'Chal_Acc': f"{sub['chal_acc'].mean():.4f} ± {sub['chal_acc'].std(ddof=1):.4f}",
        'Test_Macro_F1': f"{sub['test_macro_f1'].mean():.4f} ± {sub['test_macro_f1'].std(ddof=1):.4f}",
        'Test_Acc': f"{sub['test_acc'].mean():.4f} ± {sub['test_acc'].std(ddof=1):.4f}",
    }
    if s == 'none':
        row['Chal_Delta'] = "ref"
        row['Chal_Holm_p'] = "-"
        row['Chal_Sig'] = "-"
        row['Test_Delta'] = "ref"
        row['Test_Holm_p'] = "-"
        row['Test_Sig'] = "-"
    else:
        info = strat_lookup[s]
        row['Chal_Delta'] = f"{info['chal_delta_mean']:+.4f} ± {info['chal_delta_sd']:.4f}"
        row['Chal_Holm_p'] = f"{info['chal_holm_p']:.4f}"
        row['Chal_Sig'] = sig_stars(info['chal_holm_p'])
        row['Test_Delta'] = f"{info['test_delta_mean']:+.4f} ± {info['test_delta_sd']:.4f}"
        row['Test_Holm_p'] = f"{info['test_holm_p']:.4f}"
        row['Test_Sig'] = sig_stars(info['test_holm_p'])
    summary_rows.append(row)

summary_table = pd.DataFrame(summary_rows)
print("\n" + "="*80)
print("📊 TorchMLP Multi-Seed Performance Summary (Mean ± SD across 5 seeds)")
print("="*80)
print(summary_table.to_string(index=False))

summary_csv = os.path.join(PHASE4_DIR, 'torchmlp_multiseed_summary.csv')
summary_table.to_csv(summary_csv, index=False)
if os.path.abspath('phase4') != os.path.abspath(PHASE4_DIR):
    summary_table.to_csv('phase4/torchmlp_multiseed_summary.csv', index=False)

# Compute Per-Class Emotion Gain for TorchMLP SMOTE (rho=0.10) vs Baseline
smote_df = multiseed_df[multiseed_df['strategy'] == 'smote'].set_index('seed')
perclass_rows = []
raw_class_p_chal, raw_class_p_test = [], []
for emo in EMOTION_LABELS:
    _, p_c = stats.ttest_rel(smote_df[f'chal_f1_{emo}'], base_df[f'chal_f1_{emo}'])
    _, p_t = stats.ttest_rel(smote_df[f'test_f1_{emo}'], base_df[f'test_f1_{emo}'])
    raw_class_p_chal.append(p_c)
    raw_class_p_test.append(p_t)

holm_all_chal = holm_bonferroni(raw_class_p_chal)
holm_all_test = holm_bonferroni(raw_class_p_test)
tail_indices = [EMOTION_LABELS.index(e) for e in ['Sadness', 'Disgust', 'Surprise', 'Fear']]
tail_p_raw_chal = [raw_class_p_chal[i] for i in tail_indices]
holm_tail_chal = holm_bonferroni(tail_p_raw_chal)
tail_lookup_chal = {['Sadness', 'Disgust', 'Surprise', 'Fear'][i]: holm_tail_chal[i] for i in range(4)}

for i, emo in enumerate(EMOTION_LABELS):
    d_c = smote_df[f'chal_f1_{emo}'] - base_df[f'chal_f1_{emo}']
    d_t = smote_df[f'test_f1_{emo}'] - base_df[f'test_f1_{emo}']
    p_c = raw_class_p_chal[i]
    h_c_all = holm_all_chal[i]
    h_c_tail = tail_lookup_chal.get(emo, np.nan)
    
    perclass_rows.append({
        'Emotion': emo,
        'Chal_Base': f"{base_df[f'chal_f1_{emo}'].mean():.4f} ± {base_df[f'chal_f1_{emo}'].std(ddof=1):.4f}",
        'Chal_SMOTE': f"{smote_df[f'chal_f1_{emo}'].mean():.4f} ± {smote_df[f'chal_f1_{emo}'].std(ddof=1):.4f}",
        'Chal_Delta': f"{d_c.mean():+.4f} ± {d_c.std(ddof=1):.4f}",
        'Chal_Raw_p': f"{p_c:.4f}",
        'Chal_Holm_p_All': f"{h_c_all:.4f}",
        'Chal_Holm_p_Tail': f"{h_c_tail:.4f}" if not np.isnan(h_c_tail) else "-",
        'Chal_Sig_All': sig_stars(h_c_all),
        'Chal_Sig_Tail': sig_stars(h_c_tail) if not np.isnan(h_c_tail) else "-",
        'Test_Delta': f"{d_t.mean():+.4f} ± {d_t.std(ddof=1):.4f}",
        'Test_Holm_p': f"{holm_all_test[i]:.4f}",
        'Test_Sig': sig_stars(holm_all_test[i])
    })

perclass_table = pd.DataFrame(perclass_rows)
print("\n" + "="*80)
print("🎯 TorchMLP SMOTE (ρ=0.10) Per-Class Emotion Gain (Mean ± SD & Holm-p)")
print("="*80)
print(perclass_table.to_string(index=False))

perclass_csv = os.path.join(PHASE4_DIR, 'torchmlp_multiseed_perclass.csv')
perclass_table.to_csv(perclass_csv, index=False)
if os.path.abspath('phase4') != os.path.abspath(PHASE4_DIR):
    perclass_table.to_csv('phase4/torchmlp_multiseed_perclass.csv', index=False)

print("\nMulti-seed evaluation complete and exported!")


## ✅ Benchmark Data Generation Complete!

All 42 experimental sweep conditions and 25 multi-seed stochasticity conditions have been evaluated and checkpointed to disk:
- **Condition Checkpoints**: `cache/phase4/strat_{strategy}_rho{rho}__{clf}.json`
- **Multi-Seed Checkpoints**: `cache/phase4/strat_{strategy}_rho{rho}__mlp_seed{seed}.json`
- **Consolidated Benchmark CSV**: `cache/phase4/phase4_sweep_summary.csv`
- **Multi-Seed Summary CSVs**: `cache/phase4/torchmlp_multiseed_summary.csv` & `torchmlp_multiseed_perclass.csv`
- **Inference Predictions for Statistical Significance (Figures 8 & 9)**: `cache/phase4/predictions_linearsvc_rho025.json` and `cache/phase4/predictions_mlp_rho01.json`

---

### ➡️ Next Step: Open `phase_4_analysis.ipynb`
All empirical analysis, gain curves, McNemar significance, Stratified Bootstrap intervals, and Multi-Seed stochasticity visualizations (Figures 1–10) are located in the companion analysis notebook:  
**[`phase_4_analysis.ipynb`](./phase_4_analysis.ipynb)**.

Open `phase_4_analysis.ipynb` to generate all charts instantly without re-running the GPU sweep.